<a href="https://colab.research.google.com/github/muazzamasim04-ops/evolvix-aiml-internship-muazzamasim/blob/main/Week_4_Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Install required packages
!pip install -q fastapi uvicorn pydantic scikit-learn joblib nest-asyncio
!npm install -g localtunnel

import joblib
import numpy as np
import nest_asyncio
import asyncio
import threading
import time
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# 2. Train and save a sample ML model ('model.pkl')
iris = load_iris()
X, y = iris.data, iris.target

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)
joblib.dump(model, "model.pkl")
print("✅ Model trained and saved as 'model.pkl'")

# 3. Define FastAPI App with input validation & error handling
app = FastAPI(
    title="Model Serving API",
    description="REST API to serve a trained ML model for predictions.",
    version="1.0.0"
)

# Input Schema Validation
class PredictionInput(BaseModel):
    feature1: float = Field(..., description="Numerical feature 1", json_schema_extra={"example": 5.1})
    feature2: float = Field(..., description="Numerical feature 2", json_schema_extra={"example": 3.5})
    feature3: float = Field(..., description="Numerical feature 3", json_schema_extra={"example": 1.4})
    feature4: float = Field(..., description="Numerical feature 4", json_schema_extra={"example": 0.2})

@app.get("/")
def health_check():
    """Health check endpoint to verify API status."""
    return {"status": "online", "message": "ML Model API is running."}

@app.post("/predict")
def predict(data: PredictionInput):
    """Prediction endpoint that accepts JSON and returns the prediction."""
    try:
        loaded_model = joblib.load("model.pkl")
    except Exception as e:
        raise HTTPException(status_code=500, detail="Model file not found on server.")

    try:
        input_features = np.array([[data.feature1, data.feature2, data.feature3, data.feature4]])
        prediction = loaded_model.predict(input_features)

        return {
            "success": True,
            "prediction": int(prediction[0]) if hasattr(prediction[0], 'item') else prediction[0]
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Error processing prediction: {str(e)}")

# 4. Start Uvicorn server in a separate background thread (Python 3.13 Compatible)
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

def run_server():
    asyncio.run(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Wait briefly for server to boot up
time.sleep(2)

# 5. Expose API via LocalTunnel
get_ipython().system_raw('lt --port 8000 > url.txt 2>&1 &')
time.sleep(3)

with open('url.txt') as f:
    tunnel_url = f.read().strip()
    print(f"\n🚀 Public API URL for Postman: {tunnel_url}")
    print(f"📖 Interactive Swagger Docs: {tunnel_url}/docs")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼✅ Model trained and saved as 'model.pkl'


INFO:     Started server process [3689]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



🚀 Public API URL for Postman: your url is: https://smooth-parts-warn.loca.lt
📖 Interactive Swagger Docs: your url is: https://smooth-parts-warn.loca.lt/docs
